In [1]:
import pandas as pd

# Carregar o arquivo original
df = pd.read_parquet("03_PENSE2024_MICRODADOS_COM_LABELS.parquet")

# Exibir dimensões originais
print("Dimensões originais:", df.shape)

# Remover linhas com um ou mais valores faltantes
df_limpo = df.dropna()

# Exibir dimensões após a limpeza
print("Dimensões após limpeza:", df_limpo.shape)

# Calcular quantidade de linhas removidas
print("Linhas removidas:", len(df) - len(df_limpo))

# Salvar em um novo arquivo Parquet
df_limpo.to_parquet("04_PENSE2024_SEM_NAN.parquet", index=False)

print("Arquivo salvo com sucesso!")

Dimensões originais: (153870, 30)
Dimensões após limpeza: (148736, 30)
Linhas removidas: 5134
Arquivo salvo com sucesso!


In [14]:
import pandas as pd

# 1. Carrega as bases de dados (Dicionário + Microdados em Parquet)
df_dict = pd.read_excel('00_Dicionário_Microdados_PENSE2024.xlsx', engine='openpyxl')
df = pd.read_parquet('04_PENSE2024_SEM_NAN.parquet')  # <--- MUDANÇA AQUI!

normalizar = lambda t: "".join(str(t).split()).lower()

# 2. Cria o mapa estruturado: Código/Nome da Variável -> {Código Resposta: Significado}
mapeamento = {}
for _, row in df_dict.iterrows():
    chave = normalizar(row['Descrição da Variável'])
    cats = str(row['Categoria'])
    
    if cats and cats != 'nan':
        sub_mapa = {}
        for linha in cats.split('\n'):
            if ' - ' in linha:
                cod, desc = [p.strip() for p in linha.split(' - ', 1)]
                sub_mapa[cod] = desc
                try: sub_mapa[str(int(cod))] = desc
                except: pass
                try: sub_mapa[str(float(cod))] = desc
                except: pass
        mapeamento[chave] = sub_mapa

# 3. Processa o DataFrame carregado e gera a tabela
linhas_finais = []

for col in df.columns:
    col_limpa = str(col).strip()
    dist = df[col].value_counts(normalize=True, dropna=False) * 100
    mapa_col = mapeamento.get(normalizar(col_limpa), {})
    
    linhas_respostas = []
    for val, pct in dist.items():
        v_str = str(val).strip()
        if v_str.endswith('.0'): v_str = v_str[:-2]
        
        desc = "Valores Nulos / Em Branco (NaN)" if pd.isna(val) else mapa_col.get(v_str, "Significado não encontrado")
        linhas_respostas.append(f"[{val}] {desc}: {pct:.2f}%")
        
    linhas_finais.append({
        'Pergunta / Variável': col_limpa,
        'Distribuição das Respostas': "\n".join(linhas_respostas)
    })

# 4. Envia para o Clipboard (Excel/Pages)
df_consolidado = pd.DataFrame(linhas_finais)
df_consolidado.to_clipboard(index=False, sep='\t')

print("✨ Dados copiados para a área de transferência! Prontos para o Excel.\n")
print("📊 EXIBIÇÃO DOS RESULTADOS NO IPYNB:")

# 5. Exibe a tabela formatada na área de resultados do notebook
df_consolidado.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'})


✨ Dados copiados para a área de transferência! Prontos para o Excel.

📊 EXIBIÇÃO DOS RESULTADOS NO IPYNB:


,Pergunta / Variável,Distribuição das Respostas
0,"NOS ÚLTIMOS 12 MESES, alguma vez você sentiu vontade de se machucar de propósito?",[2] Não: 66.75% [1] Sim: 32.65% [-9] Sem resposta: 0.35% [-2] Abandono: 0.24%
1,"NOS ÚLTIMOS 30 DIAS, com que frequência você sentiu que a vida não vale a pena ser vivida?",[1] Nunca: 48.84% [2] Raramente: 17.36% [3] Às vezes: 15.60% [4] Na maioria das vezes: 8.80% [5] Sempre: 8.74% [-9] Sem resposta: 0.41% [-2] Abandono: 0.24%
2,"NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu triste?",[3] Às vezes: 36.24% [2] Raramente: 23.71% [4] Na maioria das vezes: 19.28% [1] Nunca: 11.07% [5] Sempre: 9.17% [-9] Sem resposta: 0.31% [-2] Abandono: 0.22%
3,"NOS ÚLTIMOS 30 DIAS, com que frequência você sentiu que ninguém se preocupa com você?",[1] Nunca: 27.77% [3] Às vezes: 24.35% [2] Raramente: 22.24% [4] Na maioria das vezes: 14.19% [5] Sempre: 10.87% [-9] Sem resposta: 0.36% [-2] Abandono: 0.22%
4,"NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu muito preocupado com as coisas comuns do seu dia a dia como atividades da escola, competições esportivas, tarefas de casa, etc.?",[4] Na maioria das vezes: 27.73% [3] Às vezes: 25.47% [5] Sempre: 25.11% [2] Raramente: 11.73% [1] Nunca: 9.39% [-9] Sem resposta: 0.37% [-2] Abandono: 0.21%
5,"NOS ÚLTIMOS 30 DIAS, com que frequência você se sentiu irritado(a), nervoso(a) ou mal-humorado(a) por qualquer coisa?",[3] Às vezes: 31.85% [4] Na maioria das vezes: 24.84% [5] Sempre: 17.76% [2] Raramente: 15.73% [1] Nunca: 9.25% [-9] Sem resposta: 0.35% [-2] Abandono: 0.23%
6,Quantos(as) amigos(as) próximos você tem?,[4] 3 ou mais amigos: 75.58% [3] 2 amigos: 13.72% [2] 1 amigo: 6.41% [1] Nenhum amigo: 3.86% [-9] Sem resposta: 0.23% [-2] Abandono: 0.20%
7,"NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável entendeu suas preocupações e problemas?",[5] Sempre: 22.94% [4] Na maioria das vezes: 21.73% [3] Às vezes: 20.27% [2] Raramente: 17.28% [1] Nunca: 17.23% [-9] Sem resposta: 0.41% [-2] Abandono: 0.15%
8,"NOS ÚLTIMOS 30 DIAS, com que frequência sua mãe, pai ou responsável sabia realmente o que você estava fazendo em seu tempo livre?",[5] Sempre: 33.90% [4] Na maioria das vezes: 33.54% [3] Às vezes: 15.27% [2] Raramente: 8.85% [1] Nunca: 7.96% [-9] Sem resposta: 0.34% [-2] Abandono: 0.14%
9,"NOS ÚLTIMOS 30 DIAS, quantas vezes algum dos seus colegas de escola o esculachou, zoou, mangou, intimidou ou caçoou tanto que você ficou magoado, incomodado, aborrecido, ofendido ou humilhado?",[1] Nenhuma vez nos últimos 30 dias: 58.37% [4] 3 ou mais vezes: 19.02% [2] 1 vez: 13.16% [3] 2 vezes: 8.92% [-9] Sem resposta: 0.37% [-2] Abandono: 0.16%
